# Data Quality & Schema Validation

**Question**: Can we trust the data as delivered? What structural issues exist, and what must be addressed before downstream analysis?

This notebook validates all three datasets against defined schemas, investigates zero values in Radnett, documents coordinate issues in Civil Defence data, and quantifies station operational reliability.


## Setup

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_radnett, load_station_locations, load_civil_defence
from src.schemas import validate_radnett, validate_stations, validate_civil_defence
from src.utils import (
    save_figure, classify_uptime, 
    STATION_TYPE_COLORS, UPTIME_COLORS
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:.4f}".format)


## 1. Load all datasets

In [16]:
radnett = load_radnett()
stations = load_station_locations()
civil_defence = load_civil_defence()

print(f"Radnett:        {radnett.shape[0]:>10,} rows × {radnett.shape[1]} cols")
print(f"Stations:       {stations.shape[0]:>10,} rows × {stations.shape[1]} cols")
print(f"Civil Defence:  {civil_defence.shape[0]:>10,} rows × {civil_defence.shape[1]} cols")


Radnett:           385,484 rows × 5 cols
Stations:               44 rows × 4 cols
Civil Defence:       2,356 rows × 13 cols


## 2. Schema Validation

In [3]:
# Validate each dataset against its Pandera schema
datasets = {
    "Radnett": (validate_radnett, radnett),
    "Station Locations": (validate_stations, stations),
    "Civil Defence": (validate_civil_defence, civil_defence),
}

for name, (validator, data) in datasets.items():
    try:
        validator(data)
        print(f"✓ {name}: schema valid")
    except Exception as e:
        print(f"✗ {name}: schema violation — {e}")


✓ Radnett: schema valid
✓ Station Locations: schema valid
✓ Civil Defence: schema valid


## 3. Radnett Data Inspection

### 3.1 Basic structure

In [4]:
print("Column types:")
print(radnett.dtypes)
print(f"\nTime range: {radnett['time'].min()} → {radnett['time'].max()}")
print(f"Unique stations: {radnett['station_name'].nunique()}")
print(f"\nStation type distribution:")
print(radnett["station_type"].value_counts())
print(f"\nDose rate summary (all values including zeros):")
print(radnett["dose_rate_microsv_h"].describe())


Column types:
station_code                    int64
station_name                      str
time                   datetime64[us]
dose_rate_microsv_h           float64
station_type                      str
dtype: object

Time range: 2023-01-01 00:00:00 → 2024-01-01 00:00:00
Unique stations: 44

Station type distribution:
station_type
fixed         297874
mobile         52566
air_filter     35044
Name: count, dtype: int64

Dose rate summary (all values including zeros):
count   385484.0000
mean         0.0731
std          0.0466
min          0.0000
25%          0.0560
50%          0.0780
75%          0.1030
max          0.2610
Name: dose_rate_microsv_h, dtype: float64


### 3.2 Zero value investigation

Zero values in dose rate data could mean actual zero radiation (physically implausible for background monitoring) or missing/failed measurements coded as zero. This distinction is critical — treating missing data as zero measurements would understate system failures and overstate coverage.

In [5]:
# Calculate zero percentage per station
total_hours = radnett.groupby("station_name").size()
zero_hours = radnett[radnett["dose_rate_microsv_h"] == 0].groupby("station_name").size()
zero_pct = (zero_hours / total_hours * 100).fillna(0).sort_values(ascending=False)

# Also get non-zero stats per station
nonzero = radnett[radnett["dose_rate_microsv_h"] > 0]
station_means = nonzero.groupby("station_name")["dose_rate_microsv_h"].mean()

zero_report = pd.DataFrame({
    "total_hours": total_hours,
    "zero_hours": zero_hours.reindex(total_hours.index, fill_value=0).astype(int),
    "zero_pct": zero_pct.reindex(total_hours.index, fill_value=0),
    "mean_when_nonzero": station_means.reindex(total_hours.index),
    "station_type": radnett.groupby("station_name")["station_type"].first(),
}).sort_values("zero_pct", ascending=False)

print("Zero value analysis per station:")
print(zero_report.to_string())


Zero value analysis per station:
                       total_hours  zero_hours  zero_pct  mean_when_nonzero station_type
station_name                                                                            
Mobil målestasjon 1           8761        8761  100.0000                NaN       mobile
Lista                         8761        8761  100.0000                NaN        fixed
Kjeller                       8761        8293   94.6581             0.1143        fixed
Skibotn (Luftfilter)          8761        8075   92.1698             0.0983   air_filter
Mobil målestasjon 6           8761        7904   90.2180             0.1582       mobile
Mobil målestasjon 3           8761        7040   80.3561             0.1772       mobile
Runde                         8761        6084   69.4441             0.1079        fixed
Haakonsvern                   8761        5262   60.0616             0.1666        fixed
Førde                         8761        4721   53.8865             0.1116  

In [6]:
# Visualize: zero percentage by station
fig, ax = plt.subplots(figsize=(14, 8))

colors = [STATION_TYPE_COLORS.get(t, "#999") for t in zero_report["station_type"]]

bars = ax.barh(
    range(len(zero_report)),
    zero_report["zero_pct"],
    color=colors, 
    edgecolor="white",
    linewidth=0.5,
)

ax.set_yticks(range(len(zero_report)))
ax.set_yticklabels(zero_report.index, fontsize=9)
ax.set_xlabel("Zero values (%)")
ax.set_title("Percentage of zero-coded readings per Radnett station (2023)")
ax.invert_yaxis()

# Add threshold lines
ax.axvline(x=50, color="red", linestyle="--", alpha=0.5, label=">50%: likely offline")
ax.axvline(x=5, color="orange", linestyle="--", alpha=0.5, label=">5%: significant gaps")

# Legend for station types
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=STATION_TYPE_COLORS["fixed"], label="Fixed"),
    Patch(facecolor=STATION_TYPE_COLORS["air_filter"], label="Air filter"),
    Patch(facecolor=STATION_TYPE_COLORS["mobile"], label="Mobile"),
    plt.Line2D([0], [0], color="red", linestyle="--", alpha=0.5, label=">50%: likely offline"),
    plt.Line2D([0], [0], color="orange", linestyle="--", alpha=0.5, label=">5%: significant gaps"),
]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
save_figure(fig, "zero_values_per_station")
plt.show()


### 3.3 Uptime classification

Based on the zero-value analysis, classify each station's operational reliability.

In [7]:
# Classify uptime
zero_report["uptime_fraction"] = 1 - zero_report["zero_pct"] / 100
zero_report["uptime_class"] = zero_report["uptime_fraction"].apply(classify_uptime)

print("Station reliability classification:")
print(zero_report[["zero_pct", "uptime_fraction", "uptime_class", "station_type"]].to_string())
print(f"\nSummary:")
print(zero_report["uptime_class"].value_counts())


Station reliability classification:
                       zero_pct  uptime_fraction uptime_class station_type
station_name                                                              
Mobil målestasjon 1    100.0000           0.0000      offline       mobile
Lista                  100.0000           0.0000      offline        fixed
Kjeller                 94.6581           0.0534      offline        fixed
Skibotn (Luftfilter)    92.1698           0.0783      offline   air_filter
Mobil målestasjon 6     90.2180           0.0978      offline       mobile
Mobil målestasjon 3     80.3561           0.1964      offline       mobile
Runde                   69.4441           0.3056      offline        fixed
Haakonsvern             60.0616           0.3994      offline        fixed
Førde                   53.8865           0.4611      offline        fixed
Østerås (Luftfilter)    51.8206           0.4818      offline   air_filter
Bodø                    38.2491           0.6175     unstable   

### 3.4 Operational availability over time

How many stations were actually operational each week? This reveals whether outages were scattered or concentrated.

In [8]:
# Weekly operational station count (excluding mobile stations for clarity)
fixed_stations = radnett[radnett["station_type"].isin(["fixed", "air_filter"])].copy()
fixed_stations["week"] = fixed_stations["time"].dt.isocalendar().week.astype(int)

# Count stations with at least one non-zero reading per week
weekly_operative = (
    fixed_stations[fixed_stations["dose_rate_microsv_h"] > 0]
    .groupby("week")["station_name"]
    .nunique()
)

total_fixed = fixed_stations["station_name"].nunique()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_operative.index, weekly_operative.values, "o-", color="#2196F3", linewidth=2)
ax.axhline(y=total_fixed, color="green", linestyle="--", alpha=0.5, 
           label=f"Total fixed + air filter stations ({total_fixed})")
ax.fill_between(weekly_operative.index, weekly_operative.values, total_fixed, 
                alpha=0.15, color="red")
ax.set_xlabel("Week number (2023)")
ax.set_ylabel("Number of operative stations")
ax.set_title("Effective number of operative stations per week (fixed + air filter only)")
ax.legend()
ax.set_ylim(0, total_fixed + 2)

plt.tight_layout()
save_figure(fig, "operative_stations_per_week")
plt.show()

print(f"Design capacity: {total_fixed} fixed/air_filter stations")
print(f"Minimum operative in any week: {weekly_operative.min()}")
print(f"Average operative per week: {weekly_operative.mean():.1f}")


Design capacity: 38 fixed/air_filter stations
Minimum operative in any week: 30
Average operative per week: 32.5


### 3.5 Station uptime heatmap

In [9]:
# Create weekly uptime matrix: stations × weeks
# 1 = had data, 0 = all zeros that week
fixed_data = radnett[radnett["station_type"].isin(["fixed", "air_filter"])].copy()
fixed_data["week"] = fixed_data["time"].dt.isocalendar().week.astype(int)

# For each station-week pair, check if there's any non-zero reading
uptime_matrix = (
    fixed_data.groupby(["station_name", "week"])["dose_rate_microsv_h"]
    .apply(lambda x: (x > 0).any().astype(int))
    .unstack(fill_value=0)
)

# Sort by total uptime
uptime_matrix = uptime_matrix.loc[uptime_matrix.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    uptime_matrix,
    cmap=["#F44336", "#4CAF50"],
    cbar=False,
    linewidths=0.3,
    linecolor="white",
    ax=ax,
)
ax.set_xlabel("Week number (2023)")
ax.set_ylabel("")
ax.set_title("Station operational status by week (green = data, red = no data)")

plt.tight_layout()
save_figure(fig, "station_uptime_heatmap")
plt.show()


## 4. Station Locations Inspection

In [10]:
print("Station location data:")
print(stations[["station_name", "latitude", "longitude"]].to_string())
print(f"\nData types:")
print(stations[["latitude", "longitude"]].dtypes)
print(f"\nAny NaN coordinates: {stations[['latitude', 'longitude']].isna().any().any()}")

# Check for suspicious coordinates
print(f"\nCoordinate ranges:")
print(f"  Latitude:  {stations['latitude'].min():.2f} to {stations['latitude'].max():.2f}")
print(f"  Longitude: {stations['longitude'].min():.2f} to {stations['longitude'].max():.2f}")

# Check for duplicate/placeholder coordinates
coord_counts = stations.groupby(["latitude", "longitude"]).size()
duplicates = coord_counts[coord_counts > 1]
if len(duplicates) > 0:
    print(f"\nDuplicate coordinates found:")
    for (lat, lon), count in duplicates.items():
        names = stations[
            (stations["latitude"] == lat) & (stations["longitude"] == lon)
        ]["station_name"].tolist()
        print(f"  ({lat:.2f}, {lon:.2f}): {count} stations — {names}")


Station location data:
             station_name  latitude  longitude
0    Østerås (Luftfilter)   59.9479    10.6029
1       Sola (Luftfilter)   58.8808     5.6459
2   Svanhovd (Luftfilter)   69.4551    30.0409
3                 Svolvær   68.2322    14.5626
4    Skibotn (Luftfilter)   69.3717    20.3038
5                   Runde   62.3990     5.6607
6                 Arendal   58.4764     8.8178
7                    Oslo   59.9431    10.7211
8                Svanhovd   69.4552    30.0412
9              Hammerfest   70.6713    23.6660
10                 Tromsø   69.6537    18.9366
11               Karasjok   69.4635    25.5025
12           Longyearbyen   78.2226    15.6244
13                Kjeller   59.9761    11.0501
14                  Hitra   63.6385     8.6873
15             Kautokeino   69.0219    23.0193
16            Haakonsvern   60.3355     5.2345
17              Sørkjosen   69.7856    20.9415
18    Mobil målestasjon 1   59.9479    10.6028
19    Mobil målestasjon 2   60.0000  

## 5. Civil Defence Data Inspection

### 5.1 Basic structure

In [11]:
print(f"Time range: {civil_defence['timestamp'].min()} → {civil_defence['timestamp'].max()}")
print(f"Unique measurement points: {civil_defence['measurement_point_name'].nunique()}")
print(f"\nMeasurement type: {civil_defence['measurement_type'].unique()}")
print(f"Events: {civil_defence['event'].unique()}")
print(f"\nDose rate summary (µSv/h):")
print(civil_defence["dose_rate_microsv_h"].describe())
print(f"\nMeasurement height values: {sorted(civil_defence['measurement_height'].dropna().unique())}")
print(f"\nRainfall flag distribution:")
print(civil_defence["rainfall"].value_counts())


Time range: 2002-06-10 06:15:00+00:00 → 2024-06-13 10:51:00+00:00
Unique measurement points: 465

Measurement type: <StringArray>
['FOT']
Length: 1, dtype: str
Events: <StringArray>
['Sivilforsvaret']
Length: 1, dtype: str

Dose rate summary (µSv/h):
count   2356.0000
mean       0.0727
std        0.0931
min        0.0000
25%        0.0560
50%        0.0680
75%        0.0820
max        4.3500
Name: dose_rate_microsv_h, dtype: float64

Measurement height values: [np.float64(1.0), np.float64(1.5)]

Rainfall flag distribution:
rainfall
False    1924
True      431
Name: count, dtype: int64


### 5.2 Coordinate quality

In [12]:
# Check coordinate ranges
print("Coordinate ranges:")
print(f"  Latitude:  {civil_defence['latitude'].min():.4f} to {civil_defence['latitude'].max():.4f}")
print(f"  Longitude: {civil_defence['longitude'].min():.4f} to {civil_defence['longitude'].max():.4f}")

# Norway's bounding box is approximately: lat 57.5-71.5, lon 4.5-31.5
# Svalbard extends to ~80°N, ~34°E

# Flag suspect coordinates
suspect = civil_defence[
    (civil_defence["latitude"] < 55) | 
    (civil_defence["latitude"] > 82) |
    (civil_defence["longitude"] < 3) |
    (civil_defence["longitude"] > 35)
]

print(f"\nEntries with coordinates outside Norway/Svalbard bounds: {len(suspect)}")
if len(suspect) > 0:
    print(suspect[["latitude", "longitude", "measurement_point_name", 
                    "dose_rate_microsv_h", "timestamp"]].to_string())


Coordinate ranges:
  Latitude:  0.0000 to 71.0642
  Longitude: 0.0000 to 57.2413

Entries with coordinates outside Norway/Svalbard bounds: 13
      latitude  longitude          measurement_point_name  dose_rate_microsv_h                 timestamp
400     0.0000     0.0000                Klungset, Fauske               0.0590 2022-05-02 09:53:00+00:00
953    60.2653     2.7898            Uvdal, Nore og Uvdal               0.0670 2022-12-21 11:55:00+00:00
972     0.0000     0.0000                Horn Brønnøysund               0.0450 2023-01-19 10:05:00+00:00
1062    1.8306    57.1619                  Leirvik, Stord               0.0820 2023-02-06 11:30:00+00:00
1111    1.7761    57.2413               Rimbareid, Fitjar               0.0850 2023-02-06 12:40:00+00:00
1330    0.0000     0.0000  Båsmo barnehage parkering nord               0.0550 2023-05-02 06:27:00+00:00
1524   60.2653     2.7898            Uvdal, Nore og Uvdal               0.0780 2023-06-14 09:56:00+00:00
1584    0.0000    

### 5.3 Extreme values

In [13]:
# Look at the distribution and flag potential outliers
print("Dose rate percentiles (µSv/h):")
for p in [50, 75, 90, 95, 99, 99.5, 100]:
    val = civil_defence["dose_rate_microsv_h"].quantile(p / 100)
    print(f"  {p:5.1f}th: {val:.4f}")

# Show top values
print(f"\nTop 10 highest readings:")
top10 = civil_defence.nlargest(10, "dose_rate_microsv_h")[
    ["measurement_point_name", "dose_rate_microsv_h", "timestamp", "latitude", "longitude", "rainfall"]
]
print(top10.to_string())


Dose rate percentiles (µSv/h):
   50.0th: 0.0680
   75.0th: 0.0820
   90.0th: 0.0975
   95.0th: 0.1110
   99.0th: 0.1580
   99.5th: 0.1782
  100.0th: 4.3500

Top 10 highest readings:
               measurement_point_name  dose_rate_microsv_h                 timestamp  latitude  longitude rainfall
189                               NaN               4.3500 2023-04-20 11:48:00+00:00   67.0689    14.2588     None
1611           Storvannet, HAmmerfest               0.7000 2023-10-18 10:30:00+00:00   70.6602    23.7207    False
1784                  Halne, Eidfjord               0.5500 2023-09-04 07:30:00+00:00   60.4172     7.6853    False
1051  Caravan-plass, Strand, Sortland               0.2800 2022-12-15 10:45:00+00:00   68.7083    15.4417    False
1052           Sortland, Vikveien 347               0.2400 2022-12-15 10:00:00+00:00   68.7785    15.3058    False
1577                 Svortland, Bømlo               0.2180 2023-11-02 11:53:00+00:00   59.7954     5.1819    False
1050        

### 5.4 Unit consistency check

The two datasets use different units. Verify the conversion is correct by comparing typical ranges.

In [14]:
# Compare typical values across datasets
radnett_nonzero = radnett[radnett["dose_rate_microsv_h"] > 0]["dose_rate_microsv_h"]
cd_values = civil_defence["dose_rate_microsv_h"]

print("Typical dose rate ranges (µSv/h):")
print(f"  Radnett median:         {radnett_nonzero.median():.4f}")
print(f"  Civil Defence median:   {cd_values.median():.4f}")
print(f"  Radnett mean:           {radnett_nonzero.mean():.4f}")
print(f"  Civil Defence mean:     {cd_values.mean():.4f}")
print(f"  Radnett 5-95th:         {radnett_nonzero.quantile(0.05):.4f} – {radnett_nonzero.quantile(0.95):.4f}")
print(f"  Civil Defence 5-95th:   {cd_values.quantile(0.05):.4f} – {cd_values.quantile(0.95):.4f}")

print("\nIf units are correctly aligned, medians should be in the same order of magnitude (0.05–0.15 µSv/h).")


Typical dose rate ranges (µSv/h):
  Radnett median:         0.0890
  Civil Defence median:   0.0680
  Radnett mean:           0.0942
  Civil Defence mean:     0.0727
  Radnett 5-95th:         0.0590 – 0.1460
  Civil Defence 5-95th:   0.0400 – 0.1110

If units are correctly aligned, medians should be in the same order of magnitude (0.05–0.15 µSv/h).


## Key Findings

*To be filled after running the notebook — findings should come from the data, not be pre-written.*

## Take-Home Message

*One sentence summary of what this means for DSA, to be written after analysis.*

## Acceptance Criteria

- [ ] All three datasets load without errors
- [ ] Schema validation runs and results documented
- [ ] Zero values in Radnett quantified per station
- [ ] Stations classified by operational reliability
- [ ] Effective station count over time calculated and plotted
- [ ] Station uptime heatmap generated
- [ ] Station location coordinate issues identified
- [ ] Civil Defence coordinate quality assessed
- [ ] Extreme values in Civil Defence documented
- [ ] Unit consistency between datasets verified
